# DSC 210 Exam 2: Take-home portion

**DSC 210 Foundations of Data Science**

Covers Modules 8 to 11. Worth 30 of the exam's 100 points.

```
ASK  ->  GET  ->  [ EXPLORE ]  ->  MODEL  ->  COMMUNICATE
```

References:
- Data: "fishcatch," submitted by Juha Puranen (University of Helsinki) to the *Journal of Statistics Education* data archive, https://jse.amstat.org/datasets/fishcatch.txt (description) and `fishcatch.dat` (data). Original source: Pekka Brofeldt (1917), "Bidrag till kännedom om fiskbeståndet i våra sjöar. Längelmävesi," in T. H. Järvi, *Finlands Fiskerier*, Band 4. Our copy, `csv_data/fishcatch.csv`, keeps the archive's values and codes.

## Instructions

**Total: 30 points.** Plan on about 1.5 to 2 hours.

- This is an exam: work **alone**. You may use your course notebooks, your homework, and the pandas, seaborn, and scikit-learn documentation. You may not discuss the exam with anyone or use a generative AI tool.
- Use only methods from the course. Cells marked `RUN-TOGETHER` are complete: run them and read the output. Cells marked with a problem number are yours to fill in.
- Every figure needs a title and labeled axes (the figures we draw for you already have them).
- Most of the credit rides on your written answers. **Quote specific numbers from your output.** One or two specific sentences beat a long, general paragraph.
- Submit the completed `.ipynb`, with all cells run and all figures visible, to Canvas.

## The scenario and the data

In 1917 a Finnish researcher caught and measured 159 fish of seven species from Längelmävesi, a lake near Tampere, Finland. A fisheries office wants to know three things: **do the species fall into natural body types, can a fish's species be identified from its measurements, and how well can a fish's weight be predicted from its length?** (In the field, length is quick to measure; weight needs a scale and a calm boat.)

| Column | Meaning |
| --- | --- |
| `obs` | fish number, 1 to 159 |
| `species` | species code, 1 to 7 (names are in a separate table) |
| `weight` | weight (grams) |
| `length1` | length from the nose to the beginning of the tail (cm) |
| `length2` | length from the nose to the notch of the tail (cm) |
| `length3` | length from the nose to the end of the tail (cm) |
| `height_pct` | greatest body height, **as a percentage of `length3`** |
| `width_pct` | greatest body width, **as a percentage of `length3`** |
| `sex` | 1 = male, 0 = female |

A note on names: the source lists species 4 (*Abramis bjoerkna*) without an English name; we use its Finnish name, *parkki* (in English it is usually called the silver bream). The source calls species 2 "whitefish" but gives the Latin name *Leuciscus idus*, a fish usually called the ide in English. We keep the source's name.

In [ ]:
# RUN-TOGETHER  (setup)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/'
fish = pd.read_csv(url + 'fishcatch.csv')

species = pd.DataFrame({
    'species': [1, 2, 3, 4, 5, 6, 7],
    'name':    ['Bream', 'Whitefish', 'Roach', 'Parkki', 'Smelt', 'Pike', 'Perch'],
    'latin':   ['Abramis brama', 'Leuciscus idus', 'Leuciscus rutilus', 'Abramis bjoerkna',
                'Osmerus eperlanus', 'Esox lucius', 'Perca fluviatilis'],
})
print('fish:', fish.shape)
fish.head()

---
## Problem 1. Getting the fish ready (6 points)
---

**(a) (2 points)** Merge `fish` with `species` so that every fish gets its species `name`, and store the result back in `fish`. Which join type did you choose, and why is it the safe choice here? Confirm that no fish was lost or left without a name, and report how many fish of each species there are.

In [ ]:
# Problem 1(a): your code here


**Your answer:**



**(b) (2 points)** Find the missing values in every column. Then look for a **disguised** missing value in `weight` (hint: `describe()`, then a boolean mask). Which fish are affected, and what should be done with them **before Problem 4, where weight is the label we predict**? Explain why imputing their weights would be a poor choice there.

In [ ]:
# Problem 1(b): your code here


**Your answer:**



**(c) (2 points)** Make a crosstab of species `name` against whether `sex` is missing. Is the missingness in `sex` random or systematic? Explain what the pattern suggests about how the data were recorded, and decide whether `sex` should be used as a feature in this exam.

In [ ]:
# Problem 1(c): your code here


**Your answer:**



The cell below records our cleaning decisions and builds the table used in the rest of the exam: it turns the disguised 0 into `NaN`, sets aside the 2 fish with no usable weight, and converts height and width from percentages of `length3` into centimeters (the formula comes with the data).

In [ ]:
# RUN-TOGETHER  (cleaning decisions)
fish.loc[fish['weight'] == 0, 'weight'] = np.nan          # Decision 1: a 0 g fish is a disguised missing value
fish_clean = fish.dropna(subset=['weight']).copy()        # Decision 2: set aside the 2 fish with no usable weight
fish_clean['height_cm'] = fish_clean['height_pct'] * fish_clean['length3'] / 100   # Decision 3: percentages to cm
fish_clean['width_cm'] = fish_clean['width_pct'] * fish_clean['length3'] / 100
                                                          # Decision 4: sex is not used (Problem 1c)
print('fish in fish_clean:', len(fish_clean))

---
## Problem 2. Body shape or body size? (8 points)
---

We now hide the species names and ask what structure the measurements contain on their own. There are two kinds of measurement here:

- **shape:** `height_pct` and `width_pct` describe the *proportions* of a fish, whatever its size;
- **size:** `length3`, `height_cm`, and `width_cm` grow as a fish grows.

In [ ]:
# RUN-TOGETHER  (setup for Problem 2)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

shape_features = ['height_pct', 'width_pct']
size_features = ['length3', 'height_cm', 'width_cm']
X_shape = StandardScaler().fit_transform(fish_clean[shape_features])
X_size = StandardScaler().fit_transform(fish_clean[size_features])

**(a) (2 points)** Draw an elbow plot of WCSS for k-means on `X_shape`, for k = 1 to 7 (use `n_init=10, random_state=210`). Which k does it suggest? Support your choice with the drops in WCSS.

In [ ]:
# Problem 2(a): your code here


**Your answer:**



**(b) (2 points)** Fit k-means with **k = 3** on `X_shape`, store the labels in `fish_clean['shape_cluster']`, and crosstab species `name` against the clusters. Also print the mean `height_pct` and `width_pct` of each cluster. Describe each cluster as a **body type** and say which species it contains.

In [ ]:
# Problem 2(b): your code here


**Your answer:**



**(c) (2 points)** Run the cell below, which repeats k-means with k = 3 on the **size** features instead. Do these clusters match species? What did k-means find instead, and why? Connect your answer to the penguin case study ("species, sex, or size?").

In [ ]:
# RUN-TOGETHER  (Problem 2c)
km_size = KMeans(n_clusters=3, n_init=10, random_state=210).fit(X_size)
fish_clean['size_cluster'] = km_size.labels_
print(pd.crosstab(fish_clean['name'], fish_clean['size_cluster']))
print()
print(fish_clean.groupby('size_cluster')[['length3', 'weight']].mean().round(0))

**Your answer:**



**(d) (2 points)** Fit PCA to the six scaled measurements `['weight', 'length1', 'length2', 'length3', 'height_cm', 'width_cm']` and print the share of variance for each component and the PC1 and PC2 loadings. Then run the cell after yours, which shows each species' average PC1 and PC2 scores.

1. PC1 alone explains a very large share. Name PC1, and explain why one component can summarize six measurements so well.
2. Name PC2 from its loadings, and say which species score highest and lowest on it. Does that agree with part (b)?

In [ ]:
# Problem 2(d): your code here (create pca and pcs)


In [ ]:
# RUN-TOGETHER  (Problem 2d: average scores by species; needs pcs from your cell)
scores = pd.DataFrame({'name': fish_clean['name'].to_numpy(), 'PC1': pcs[:, 0], 'PC2': pcs[:, 1]})
print(scores.groupby('name')[['PC1', 'PC2']].mean().round(2).sort_values('PC2'))

**Your answer:**



---
## Problem 3. Which species is it? (8 points)
---

A field technician measures a fish and wants software to name its species. Features: `weight`, `length3`, `height_pct`, `width_pct` (a mix of size and shape). Label: `name`.

In [ ]:
# RUN-TOGETHER  (setup for Problem 3)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

features = ['weight', 'length3', 'height_pct', 'width_pct']
X = fish_clean[features]
y = fish_clean['name']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=210, stratify=y)
print('training fish:', len(X_train), '  test fish:', len(X_test))
print(y_test.value_counts())

**(a) (1 point)** What test accuracy does the study map's baseline (always guess the most common species) achieve?

In [ ]:
# Problem 3(a): your code here


**Your answer:**



**(b) (2 points)** Fit k-NN with `k = 5` on the raw features and on scaled features (fit the scaler on the **training** data only). Report both test accuracies and explain the difference. Which feature controls the unscaled distances, and why does that hurt here?

In [ ]:
# Problem 3(b): your code here


**Your answer:**



**(c) (2 points)** Fit a decision tree with no `max_depth` (`random_state=0`) and report its training and test accuracy. Then fit a random forest (`n_estimators=300, random_state=0`) and report its test accuracy. Explain the tree's gap between training and test accuracy, and explain how the forest's **two sources of randomness** and its **majority vote** address that weakness.

In [ ]:
# Problem 3(c): your code here


**Your answer:**



**(d) (3 points)** Print the random forest's confusion matrix with `labels=labels` (defined in the cell), and its overall test accuracy. Then:

1. Compute, from the matrix, the **recall** for whitefish and for parkki, and the **precision** for bream.
2. Which species are confused with each other? Use your answers to Problem 2(b) to explain why those mistakes are the ones you would expect.
3. Explain why the overall accuracy hides the model's weakness on rare species.

In [ ]:
# Problem 3(d): your code here
labels = ['Bream', 'Parkki', 'Perch', 'Pike', 'Roach', 'Smelt', 'Whitefish']


**Your answer:**



---
## Problem 4. How heavy is a fish? (8 points)
---

Weight is the label now, so we use `fish_clean`, which already excludes the two fish without a usable weight.

In [ ]:
# RUN-TOGETHER  (setup for Problem 4)
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xr = fish_clean[['length3']]
yr = fish_clean['weight']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=210)
print('length3 ranges from', Xr['length3'].min(), 'to', Xr['length3'].max(), 'cm')

**(a) (2 points)** Fit a linear regression of `weight` on `length3` using the training data. Report the slope and intercept and interpret the slope in context, with units. Then use the line to predict the weight of a **10 cm** fish. Explain what is wrong with that prediction, and why it is a different problem from the intercept being impossible.

In [ ]:
# Problem 4(a): your code here


**Your answer:**



**(b) (2 points)** Evaluate the line on the test set (MAE, RMSE, $R^2$). Then run the residual-plot cell. Describe the pattern, and explain why a model with this $R^2$ can still be the wrong shape. (Think about how weight should grow as a fish of the same proportions gets longer.)

In [ ]:
# Problem 4(b): your code here (metrics)


In [ ]:
# RUN-TOGETHER  (Problem 4b: residuals; needs line from 4a)
resid = pd.DataFrame({'length3': Xr_test['length3'],
                      'residual': yr_test - line.predict(Xr_test)})
sns.scatterplot(data=resid, x='length3', y='residual')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Residuals of the line, test fish')
plt.xlabel('length3 (cm)'); plt.ylabel('actual minus predicted weight (g)')
plt.show()

bands = pd.cut(resid['length3'], [0, 20, 30, 40, 50, 70])
print(resid.groupby(bands, observed=True)['residual'].agg(['mean', 'count']).round(0))

**Your answer:**



**(c) (4 points)** The cell below does two things. It draws the fits of four models that use `length3` alone, and it compares the same four kinds of model using **three** features (`length3`, `height_cm`, `width_cm`) with shuffled 5-fold cross-validation on all 157 fish. Describe the shape of each fit, then choose a model for the fisheries office using the cross-validation results. Justify your choice with the mean **and** the spread of the fold scores.

In [ ]:
# RUN-TOGETHER  (Problem 4c: about 15 seconds)
models = {'line': LinearRegression(),
          'k-NN, k = 5': KNeighborsRegressor(n_neighbors=5),
          'tree, max_depth = 3': DecisionTreeRegressor(max_depth=3, random_state=0),
          'random forest': RandomForestRegressor(n_estimators=200, random_state=0)}

grid_len = pd.DataFrame({'length3': np.linspace(Xr['length3'].min(), Xr['length3'].max(), 300)})
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, (name, m) in zip(axes, models.items()):
    m.fit(Xr_train, yr_train)
    ax.scatter(Xr_train['length3'], yr_train, s=6, color='lightgray')
    ax.plot(grid_len['length3'], m.predict(grid_len), color='crimson')
    ax.set_title(name); ax.set_xlabel('length3 (cm)')
axes[0].set_ylabel('weight (g)')
plt.tight_layout()
plt.show()

X3 = fish_clean[['length3', 'height_cm', 'width_cm']]
kf = KFold(5, shuffle=True, random_state=210)
for name, m in models.items():
    s = cross_val_score(m, X3, yr, cv=kf, scoring='r2')
    print(f'{name:>20}: mean R2 = {s.mean():.3f}   folds {np.round(s, 3)}')

**Your answer:**



---
*End of the take-home portion. Before submitting: Kernel > Restart & Run All, then check that every figure appears and every answer cell is filled in.*